# HedgeManager Demo

This is the main course after the forward carry and portfolio demo. The document slightly focus more on the design pattern and would argue it add flexibility of testing new hedging strategy without killing your analyst or developer.

**Objective**: End-to-end FX hedging workflow for multi-currency portfolios using the [Strategy Pattern](https://refactoring.guru/design-patterns/strategy)

**Architecture** :
```
┌─────────────────────────────────────────────────────────────┐
│                    HedgeManager                             │
│  • Orchestrates hedging workflow                            │
│  • Generates trade-level data (exposures, carry, pricing)   │
│  • Output: Trade data only (no strategy metadata)           │
└─────────────────────────────────────────────────────────────┘
                        │ delegates to
                        ▼
┌─────────────────────────────────────────────────────────────┐
│                 HedgingStrategy                             │
│  • Calculates hedge ratios (may use carry, volatility, etc.)│
│  • Provides optional metadata (confidence, ranges)          │
│  • Pluggable: switch strategies at runtime                  │
└─────────────────────────────────────────────────────────────┘
```

**Workflow**:
```
Portfolio NAV → Identify Exposures → Strategy Calculates Ratios → 
Generate Trades → Calculate Notionals → Execute Hedges
```

**Strategy Inputs** (what strategies may use):
- **CarryCostStrategy**: Uses forward carry (premium/discount) to adjust ratios
- **ConstantRatioStrategy**: Uses pre-defined ratios (no market data needed)
- **PolicyDrivenStrategy**: Uses policy rules (no market data needed)
- **QuantileBasedStrategy**: Uses historical carry distributions

**Key Features**:
- **Strategy Pattern**: Pluggable hedging strategies (v2.5.0+)
- **Auto-filtering**: Automatically handles direct (EURUSD) and indirect (USDJPY) quotations
- **Clean Output**: `generate_trades()` returns trade data only (no strategy metadata)
- **Flexible Ratios**: Different strategies use different inputs
- **Trade History**: Backtesting support with `generate_trade_history()`

**Note on Data Timeliness**:
For institutions with private assets or hedge fund allocations not in SMA format, NAV reporting may have publication delays. This implementation assumes perfectly up-to-date data and does not model reporting lags.

**Contact:** kou001@e.ntu.edu.sg


## 1. Setup & Dependencies

In [1]:
import os
import sys
from pprint import pprint
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path so we can import src package
sys.path.insert(0, str(Path.cwd().parent))

# Import framework modules from src package
from src import Portfolio, CashFlow, RebalanceEvent
from src import ForwardCurve, load_forward_curve_config
from src import HedgeManager
# Hedging strategies (v2.5.0+)
from src import CarryCostStrategy, ConstantRatioStrategy, PolicyDrivenStrategy
# Advanced strategies (v2.7.0+)
from src import QuantileBasedStrategy, StrategyRecommendation

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("Dependencies loaded")


Dependencies loaded


## 2. Create Multi-Currency Portfolio from Existing NAV History

Setup a USD-based portfolio with exposures to EUR, GBP.

In [2]:
# Load NAV history from CSV
# Load NAV history from CSV
nav_csv_path = Path.cwd().parent / 'data' / 'raw' / 'nav_history.csv'

# Read the CSV file
nav_history_df = pd.read_csv(nav_csv_path, parse_dates=['Date'], index_col='Date')

In [3]:
# Create portfolio from NAV history
portfolio_from_nav = Portfolio.from_nav_history(
    nav_df=nav_history_df,
    home_currency='USD'
    # periodic_freq is auto-inferred from date spacing (Monthly in this case)
    # init_date, initial_nav, and weights are extracted from first row
)

print("Portfolio created from NAV history:")
print(portfolio_from_nav)
print(f"\nCurrencies: {portfolio_from_nav.currencies}")
print(f"Initial weights (inferred): {portfolio_from_nav.initial_weights}")
print(f"Periodic frequency (inferred): {portfolio_from_nav.periodic_freq}")

# NAV history is already populated - no simulation needed!
print(f"\nNAV history already loaded: {len(portfolio_from_nav.nav_history)} periods")

# Get date range from NAV history
START_DATE = nav_history_df.index[0]
END_DATE = nav_history_df.index[-1]


Portfolio created from NAV history:
Portfolio(home_currency='USD', init_date='2020-01-31', initial_nav=994,163.15, currencies=[EUR, GBP, JPY, USD], cash_flows=0, rebalances=0)

Currencies: ['EUR', 'GBP', 'JPY', 'USD']
Initial weights (inferred): {'EUR': np.float64(0.2915897349845665), 'GBP': np.float64(0.1005601731046226), 'JPY': np.float64(0.0), 'USD': np.float64(0.6078500919108109)}
Periodic frequency (inferred): M

NAV history already loaded: 60 periods


In [17]:
# Visualize Portfolio NAV History
# Use Portfolio's built-in comprehensive analysis plot
portfolio_from_nav.plot_comprehensive_analysis(
    figsize=(16, 12),
)


## 3. Load Forward Curves

Load forward curve data for each foreign currency pair.

In [5]:
# Load configuration
config_file = Path.cwd().parent / 'data' / 'raw' / 'forward_curve_tickers.csv'
config = load_forward_curve_config(config_file)

# Define currency pairs to fetch forward curves - more than what the portfolio needs
currency_pairs = ['AUDUSD', 'USDCAD', 'USDCHF', 'EURUSD', 'GBPUSD', 'USDJPY', 'USDKRW']
forward_curves = {pair: ForwardCurve.fetch(
            config=config,
            currency=pair,
            start_date=START_DATE,
            end_date=END_DATE,
            periodicity='D',
            # verbose=True
        ) for pair in currency_pairs}

print(f"\nForward curves loaded for currency pairs: {', '.join(forward_curves.keys())}")



Forward curves loaded for currency pairs: AUDUSD, USDCAD, USDCHF, EURUSD, GBPUSD, USDJPY, USDKRW


<!-- ## 4. Simulate Portfolio Returns

Generate synthetic returns and simulate portfolio NAV history. -->

In [6]:
# Demonstrate the abstract interface in action

print("="*80)
print("ABSTRACT INTERFACE DEMONSTRATION")
print("="*80)

# Show that all strategies implement the same interface
from abc import ABC
from inspect import signature

print("\n1. VERIFY INHERITANCE (all strategies inherit from HedgingStrategy ABC):")
strategies = [
    ConstantRatioStrategy({'EUR': 0.7}, default_tenor='3M'),
    CarryCostStrategy(default_tenor='3M'),
    PolicyDrivenStrategy({'EUR': 0.8}, default_tenor='3M'),
    QuantileBasedStrategy(default_tenor='3M')
]

for strategy in strategies:
    is_hedging_strategy = hasattr(strategy, 'calculate_ratios') and hasattr(strategy, 'name')
    print(f"   {strategy.name():<30} → Interface complete: {is_hedging_strategy}")

print("\n2. VERIFY METHOD SIGNATURES (all have same interface):")
for strategy in strategies:
    sig = signature(strategy.calculate_ratios)
    print(f"   {strategy.name():<30} → {sig}")

print("\n3. POLYMORPHISM IN ACTION (HedgeManager works with any strategy):")

# Test with different strategies - all work seamlessly
test_strategies = [
    ("Constant Ratio", ConstantRatioStrategy({'EUR': 0.7, 'GBP': 0.7, 'JPY': 0.5}, default_tenor='3M')),
    ("Carry Cost", CarryCostStrategy(default_tenor='3M')),
    ("Quantile Based", QuantileBasedStrategy(base_ratio=0.5, sensitivity=0.4, default_tenor='3M'))
]

for strategy_name, strategy in test_strategies:
    hm = HedgeManager(portfolio_from_nav, forward_curves, strategy=strategy)
    trades = hm.generate_trades()
    
    print(f"\n   Strategy: {strategy_name}")
    print(f"   {'Currency':<10} {'Ratio':>8} {'Rationale'}")
    print("   " + "-"*60)
    for trade in trades:
        print(f"   {trade.currency:<10} {trade.recommended_ratio:>7.0%} {trade.rationale}")

print("\n" + "="*80)
print("KEY TAKEAWAY: All strategies implement the same interface,")
print("so HedgeManager treats them uniformly (polymorphism)!")
print("="*80)

ABSTRACT INTERFACE DEMONSTRATION

1. VERIFY INHERITANCE (all strategies inherit from HedgingStrategy ABC):
   Constant Ratio Strategy        → Interface complete: True
   Carry Cost Based Strategy      → Interface complete: True
   Policy Driven Strategy         → Interface complete: True
   Quantile Based Strategy        → Interface complete: True

2. VERIFY METHOD SIGNATURES (all have same interface):
   Constant Ratio Strategy        → (context: src.hedging_strategies.HedgingContext, **kwargs) -> Dict[str, Dict[str, Any]]
   Carry Cost Based Strategy      → (context: src.hedging_strategies.HedgingContext, **kwargs) -> Dict[str, Dict[str, Any]]
   Policy Driven Strategy         → (context: src.hedging_strategies.HedgingContext, **kwargs) -> Dict[str, Dict[str, Any]]
   Quantile Based Strategy        → (context: src.hedging_strategies.HedgingContext, **kwargs) -> Dict[str, Union[Dict[str, Any], src.hedging_strategies.StrategyRecommendation]]

3. POLYMORPHISM IN ACTION (HedgeManager wo

In [7]:
# For the rest of this analysis, we'll use ConstantRatioStrategy as the primary strategy
# This demonstrates how the abstract interface works in practice

print("="*80)
print("CREATING HEDGE MANAGER WITH CONSTANTRATIOSTRATEGY")
print("="*80)

# Option 1: ConstantRatioStrategy (simple and predictable for demo)
strategy = ConstantRatioStrategy(
    ratios={'EUR': 0.7, 'GBP': 0.7, 'JPY': 0.5},
    default_ratio=0.5,
    default_tenor='3M'  # NEW in v2.6.0: Set default tenor in strategy
)

print(f"\nStrategy Selected: {strategy.name()}")
print(f"Default Tenor: {strategy.default_tenor}")
print(f"Predefined Ratios: {strategy.ratios}")
print(f"Default Ratio (for unlisted currencies): {strategy.default_ratio}")

# Create HedgeManager - it accepts ANY HedgingStrategy subclass!
hedge_manager = HedgeManager(
    portfolio=portfolio_from_nav,
    forward_curves=forward_curves,
    strategy=strategy  # Polymorphism: works with any HedgingStrategy subclass
)

print("\n" + "="*80)
print("HEDGE MANAGER INITIALIZED")
print("="*80)
print(hedge_manager)
print(f"\nBase Currency: {hedge_manager.home_currency}")
print(f"Foreign Currencies: {', '.join(hedge_manager.foreign_currencies)}")
print(f"\nActive Strategy: {hedge_manager.strategy.name()}")

print("\n" + "="*80)
print("ABSTRACT INTERFACE IN ACTION:")
print("="*80)
print("""
The HedgeManager doesn't need to know which concrete strategy class it has.
It only knows that:
1. strategy.calculate_ratios(context) will return recommendations
2. strategy.name() will return a string
3. strategy.default_tenor will provide a default tenor

This is the power of the abstract interface (Strategy Pattern)!

We can call hedge_manager.set_strategy() at ANY TIME to switch strategies:
  • hedge_manager.set_strategy(CarryCostStrategy())
  • hedge_manager.set_strategy(QuantileBasedStrategy())
  • hedge_manager.set_strategy(YourCustomStrategy())

All work seamlessly because they implement the HedgingStrategy interface!
""")

CREATING HEDGE MANAGER WITH CONSTANTRATIOSTRATEGY

Strategy Selected: Constant Ratio Strategy
Default Tenor: 3M
Predefined Ratios: {'EUR': 0.7, 'GBP': 0.7, 'JPY': 0.5}
Default Ratio (for unlisted currencies): 0.5

HEDGE MANAGER INITIALIZED
HedgeManager(home_currency='USD', foreign_currencies=3, forward_curves=3, active_hedges=0)

Base Currency: USD
Foreign Currencies: EUR, GBP, JPY

Active Strategy: Constant Ratio Strategy

ABSTRACT INTERFACE IN ACTION:

The HedgeManager doesn't need to know which concrete strategy class it has.
It only knows that:
1. strategy.calculate_ratios(context) will return recommendations
2. strategy.name() will return a string
3. strategy.default_tenor will provide a default tenor

This is the power of the abstract interface (Strategy Pattern)!

We can call hedge_manager.set_strategy() at ANY TIME to switch strategies:
  • hedge_manager.set_strategy(CarryCostStrategy())
  • hedge_manager.set_strategy(QuantileBasedStrategy())
  • hedge_manager.set_strategy(Your

## 4. HedgingStrategy Abstract Class Interface

### Understanding the Strategy Pattern

The **HedgingStrategy** abstract base class (ABC) defines a **contract** that all hedging strategies must follow. This enables the **Strategy Pattern**, a design pattern that allows algorithms to be selected at runtime.

#### Core Concept: Interface Contract

An abstract class defines **what** strategies must do, not **how** they do it:

```python
from abc import ABC, abstractmethod

class HedgingStrategy(ABC):
    """
    Abstract base class defining the interface for all hedging strategies.
    
    Subclasses MUST implement:
    - calculate_ratios(context, **kwargs) → Dict[str, StrategyOutput]
    - name() → str
    """
    
    def __init__(self, default_tenor: str = '3M'):
        self.default_tenor = default_tenor
    
    @abstractmethod
    def calculate_ratios(self, context: HedgingContext, **kwargs) -> Dict[str, StrategyOutput]:
        """
        Calculate hedge ratios for each currency.
        
        MUST return dictionary mapping currency codes to recommendations:
        {
            'EUR': {
                'recommended_ratio': 0.90,
                'rationale': 'Strong positive carry',
                'metadata': {}
            },
            'GBP': StrategyRecommendation(...)  # Enhanced output also accepted
        }
        """
        pass  # Subclasses MUST implement
    
    @abstractmethod
    def name(self) -> str:
        """Return human-readable strategy name."""
        pass  # Subclasses MUST implement
```

#### Key Benefits of Abstract Interface

1. **Polymorphism**: HedgeManager can work with ANY strategy that implements this interface
2. **Type Safety**: Python enforces that subclasses implement required methods
3. **Extensibility**: Add new strategies without modifying HedgeManager
4. **Testability**: Test strategies independently with mock data

#### The HedgingContext Input

Strategies receive a **HedgingContext** object with all decision-making data:

```python
@dataclass
class HedgingContext:
    """
    Rich data container passed to strategies.
    
    Core Data (always populated):
    - exposures: Dict mapping currencies to exposure data
    - carry_analyses: Dict mapping currencies to CarryAnalysis objects
    - tenor: Forward tenor (e.g., "3M")
    - as_of_date: Analysis date
    - exposure_type: "ASSET" or "LIABILITY"
    
    Rich Data (optional, for advanced strategies):
    - forward_curves: Full ForwardCurve objects for historical analysis
    - metadata: Strategy-specific inputs
    
    Computed Properties (lazy evaluation):
    - carry_history: DataFrame of historical carry
    - carry_quantiles: Historical percentiles for regime detection
    """
    exposures: Dict[str, Dict[str, Any]]
    carry_analyses: Dict[str, 'CarryAnalysis']
    tenor: str
    as_of_date: datetime
    exposure_type: str
    forward_curves: Optional[Dict[str, 'ForwardCurve']] = None
    metadata: Dict[str, Any] = field(default_factory=dict)
```

#### The StrategyRecommendation Output (v2.7.0+)

Strategies can return enhanced output with confidence levels and metadata:

```python
@dataclass
class StrategyRecommendation:
    """
    Enhanced strategy output structure.
    
    Required:
    - recommended_ratio: Hedge ratio (0.0 to 1.0)
    - rationale: Human-readable explanation
    
    Optional Decision Context:
    - confidence: Strategy confidence (0.0 to 1.0)
    - ratio_range: Acceptable range (min, max)
    - adjustment_factors: Breakdown of how ratio was derived
    
    Strategy-Specific Output:
    - extra_columns: Additional data for trade history analysis
    """
    recommended_ratio: float
    rationale: str
    confidence: Optional[float] = None
    ratio_range: Optional[Tuple[float, float]] = None
    adjustment_factors: Optional[Dict[str, float]] = None
    extra_columns: Dict[str, Any] = field(default_factory=dict)
```

#### Example: Simple Custom Strategy

```python
class SimpleStrategy(HedgingStrategy):
    """Always hedge 75% - simplest possible strategy."""
    
    def calculate_ratios(self, context: HedgingContext, **kwargs):
        results = {}
        for currency in context.exposures.keys():
            results[currency] = {
                'recommended_ratio': 0.75,
                'rationale': 'Fixed 75% hedge',
                'metadata': {}
            }
        return results
    
    def name(self) -> str:
        return "Simple 75% Strategy"

# Usage
strategy = SimpleStrategy()
hm = HedgeManager(portfolio, forward_curves, strategy=strategy)
trades = hm.generate_trades()  # Uses SimpleStrategy
```

#### Example: Advanced Strategy with Rich Context (v2.7.0)

```python
class AdaptiveStrategy(HedgingStrategy):
    """Adapt hedge ratio based on historical carry percentile."""
    
    def calculate_ratios(self, context: HedgingContext, **kwargs):
        results = {}
        
        # Access rich historical data
        try:
            quantiles = context.carry_quantiles  # Computed property
        except ValueError:
            quantiles = {}  # Fallback if forward_curves not available
        
        for currency in context.exposures.keys():
            pctl = quantiles.get(currency, {}).get('percentile', 0.5)
            
            # Higher carry percentile → more aggressive hedging
            ratio = 0.5 + 0.4 * (pctl - 0.5)  # Range: 0.3 to 0.7
            
            results[currency] = StrategyRecommendation(
                recommended_ratio=ratio,
                rationale=f'Carry at {pctl:.0%} percentile',
                confidence=0.8,
                ratio_range=(ratio - 0.1, ratio + 0.1),
                adjustment_factors={
                    'base_ratio': 0.5,
                    'percentile_adj': 0.4 * (pctl - 0.5)
                },
                extra_columns={
                    'carry_percentile': pctl,
                    'carry_median': quantiles.get(currency, {}).get('p50')
                }
            )
        
        return results
    
    def name(self) -> str:
        return "Adaptive Percentile Strategy"
```

#### How HedgeManager Uses the Interface

```python
class HedgeManager:
    def __init__(self, portfolio, forward_curves, strategy=None):
        # Default to CarryCostStrategy if not specified
        self.strategy = strategy or CarryCostStrategy()
    
    def generate_trades(self, tenor=None):
        # Use strategy's default tenor if not specified
        tenor = tenor or self.strategy.default_tenor
        
        # Build context with all decision-making data
        context = HedgingContext(
            exposures=self.identify_exposures(),
            carry_analyses={ccy: self.analyze_carry(...) for ccy in currencies},
            tenor=tenor,
            as_of_date=datetime.now(),
            exposure_type='ASSET',
            forward_curves=self.forward_curves,  # Pass for rich strategies
            metadata={}
        )
        
        # Delegate to strategy (polymorphism in action!)
        recommendations = self.strategy.calculate_ratios(context)
        
        # Process recommendations (handles both dict and StrategyRecommendation)
        trades = self._build_trade_data(recommendations, context)
        return trades
    
    def set_strategy(self, new_strategy: HedgingStrategy):
        """Switch strategy at runtime (Strategy Pattern benefit!)"""
        self.strategy = new_strategy
```

### Summary: Why Abstract Interface Matters

| Aspect | Without ABC | With ABC (Strategy Pattern) |
|--------|-------------|----------------------------|
| **Flexibility** | Hard-coded logic in HedgeManager | Pluggable strategies |
| **Extensibility** | Modify HedgeManager for new logic | Create new strategy class |
| **Testing** | Test entire HedgeManager | Test strategies independently |
| **Type Safety** | No enforcement of interface | Python enforces abstract methods |
| **Runtime Switching** | Not possible | `set_strategy()` anytime |

The abstract interface is the **contract** that makes the Strategy Pattern work. It defines:
- **What** strategies must provide (calculate_ratios, name)
- **What** they receive as input (HedgingContext)
- **What** they must return (Dict[str, StrategyOutput])

This allows HedgeManager to treat all strategies uniformly while each implements its own logic!


## 5. Identify Foreign Currency Exposures

Analyze current portfolio exposures by currency.

## 6. Analyze Forward Carry

Analyze the cost/benefit of hedging each currency using forward contracts.

In [8]:
# Analyze carry for each currency
tenor = '3M'

# Get exposures from HedgeManager
exposures_list = hedge_manager.identify_exposures()
# Convert to dict for easier lookup
exposures = {exp.currency: {
    'currency_pair': exp.currency_pair,
    'nav': exp.nav,
    'weight': exp.weight
} for exp in exposures_list}

print("="*80)
print(f"FORWARD CARRY ANALYSIS (Tenor: {tenor})")
print("="*80)

carry_results = {}

print(f"\n{'Pair':<10} {'Trade':<6} {'Spot':>10} {'Forward':>10} {'Points':>10} {'Carry':>12} {'Interpretation'}")
print("-"*85)

for currency in exposures.keys():
    currency_pair = exposures[currency]['currency_pair']
    
    if currency_pair not in forward_curves:
        print(f"{currency_pair:<10} {'N/A':<6} {'N/A':>10} {'N/A':>10} {'N/A':>10} {'N/A':>12} No data available")
        continue
    
    try:
        # NEW API: Must pass foreign_currency parameter
        carry = hedge_manager.analyze_carry(
            currency_pair=currency_pair,
            foreign_currency=currency,
            tenor=tenor
        )
        carry_results[currency] = carry
        
        # Use is_benefit property
        interp_short = 'BENEFIT' if carry.is_benefit else 'COST'
        
        print(f"{currency_pair:<10} {carry.trade_direction:<6} {carry.spot:>10.4f} {carry.forward_rate:>10.4f} "
              f"{carry.forward_points:>10.4f} {carry.carry_bps_ann:>11.1f} bps {interp_short}")
        
    except Exception as e:
        print(f"{currency_pair:<10} Error: {e}")

print("\n" + "="*80)
print("INTERPRETATION - 4-QUADRANT CARRY LOGIC:")
print("="*80)
print("\n1. CARRY CALCULATION:")
print("   • Carry_bps_ann considers BOTH forward premium/discount AND trade direction")
print("   • Positive carry (+) = BENEFIT: You earn from the hedge")
print("   • Negative carry (-) = COST: You pay to hedge")
print("\n2. TRADE DIRECTION (for Asset hedging):")
print("   • SELL: Selling the foreign currency forward")
print("     - EURUSD/GBPUSD: SELL = Sell EUR/GBP, Buy USD")
print("   • BUY: Buying the pair (equivalent to selling second currency)")
print("     - USDJPY: BUY = Buy USD, Sell JPY (hedges JPY exposure)")
print("\n3. FOUR-QUADRANT LOGIC:")
print("   +-----------------------------------------------+")
print("   | Trade    | Forward vs Spot | Result          |")
print("   +-----------------------------------------------+")
print("   | SELL     | Premium (fwd>sp)| BENEFIT (sell@higher) |")
print("   | SELL     | Discount (fwd<sp)| COST (sell@lower)    |")
print("   | BUY      | Premium (fwd>sp)| COST (buy@higher)    |")
print("   | BUY      | Discount (fwd<sp)| BENEFIT (buy@lower)  |")
print("   +-----------------------------------------------+")
print("\n4. EXAMPLE:")
print("   EUR asset + EURUSD spot 1.10, fwd 1.12 (EUR at premium)")
print("   → SELL EURUSD forward (sell EUR at 1.12 vs spot 1.10)")
print("   → Carry: POSITIVE (benefit of selling at higher forward rate)")
print("="*80)

FORWARD CARRY ANALYSIS (Tenor: 3M)

Pair       Trade        Spot    Forward     Points        Carry Interpretation
-------------------------------------------------------------------------------------
EURUSD     SELL       1.0354     1.0396    42.4000       164.3 bps BENEFIT
GBPUSD     SELL       1.2516     1.2507    -8.5900       -27.5 bps COST
USDJPY     BUY      157.2000   155.5506  -164.9400       420.8 bps BENEFIT

INTERPRETATION - 4-QUADRANT CARRY LOGIC:

1. CARRY CALCULATION:
   • Carry_bps_ann considers BOTH forward premium/discount AND trade direction
   • Positive carry (+) = BENEFIT: You earn from the hedge
   • Negative carry (-) = COST: You pay to hedge

2. TRADE DIRECTION (for Asset hedging):
   • SELL: Selling the foreign currency forward
     - EURUSD/GBPUSD: SELL = Sell EUR/GBP, Buy USD
   • BUY: Buying the pair (equivalent to selling second currency)
     - USDJPY: BUY = Buy USD, Sell JPY (hedges JPY exposure)

3. FOUR-QUADRANT LOGIC:
   +-----------------------------

## 7. Generate Hedge Trades (NEW API v2.6.0)

Generate hedge trade recommendations using the configured strategy.

In [9]:
# Generate hedge trade recommendations using configured strategy
# NEW API v2.6.0: generate_trades() replaces recommend_hedge_ratios()
# tenor parameter is optional - uses strategy's default_tenor if not specified

tenor = '3M'  # Can override strategy default if needed
trades = hedge_manager.generate_trades(tenor=tenor)  # Or: generate_trades() for default

print("="*80)
print(f"HEDGE TRADE RECOMMENDATIONS (Tenor: {tenor})")
print(f"Strategy: {hedge_manager.strategy.name()}")
print("="*80)

print(f"\n{'Currency':<10} {'Exposure NAV':>15} {'Trade':<6} {'Carry':>12} {'Hedge Ratio':>12} {'Rationale'}")
print("-"*95)

for trade in trades:
    carry_str = f"{trade.carry_bps_ann:+.1f} bps" if trade.carry_bps_ann is not None else "N/A"
    ratio_str = f"{trade.recommended_ratio*100:.1f}%"
    
    print(f"{trade.currency:<10} ${trade.exposure_pf_home:>14,.0f} {trade.trade_direction:<6} {carry_str:>12} {ratio_str:>12} {trade.rationale}")

print("\n" + "="*80)
print("NOTE: recommend_hedge_ratios() is deprecated - use generate_trades() instead")
print("\nCARRYCOST STRATEGY LOGIC (based on carry_bps_ann):")
print("  Carry > +100 bps → 90% hedge (strong positive carry)")
print("  Carry +20 to +100 bps → 70% hedge (moderate positive carry)")
print("  Carry -20 to +20 bps → 50% hedge (neutral)")
print("  Carry -100 to -20 bps → 30% hedge (small cost)")
print("  Carry < -100 bps → 10% hedge (high cost)")
print("="*80)

HEDGE TRADE RECOMMENDATIONS (Tenor: 3M)
Strategy: Constant Ratio Strategy

Currency      Exposure NAV Trade         Carry  Hedge Ratio Rationale
-----------------------------------------------------------------------------------------------
EUR        $     1,003,712 SELL     +164.3 bps        70.0% Constant ratio: 70%
GBP        $       472,926 SELL      -27.5 bps        70.0% Constant ratio: 70%
JPY        $             0 BUY      +420.8 bps        50.0% Constant ratio: 50%

NOTE: recommend_hedge_ratios() is deprecated - use generate_trades() instead

CARRYCOST STRATEGY LOGIC (based on carry_bps_ann):
  Carry > +100 bps → 90% hedge (strong positive carry)
  Carry +20 to +100 bps → 70% hedge (moderate positive carry)
  Carry -20 to +20 bps → 50% hedge (neutral)
  Carry -100 to -20 bps → 30% hedge (small cost)
  Carry < -100 bps → 10% hedge (high cost)


In [10]:
# Extract hedge ratios from trade recommendations
hedge_ratios = {
    trade.currency: trade.recommended_ratio
    for trade in trades
}

# Manually calculate hedge notionals from trade data
# (calculate_hedge_notionals has a bug expecting dict, not list from identify_exposures)
hedge_positions_manual = []

for trade in trades:
    # Create a hedge position object-like structure
    hedge_positions_manual.append({
        'currency_pair': trade.currency_pair,
        'direction': trade.trade_direction,
        'trade_notional_base': trade.trade_notional_base,
        'trade_notional_quote': trade.trade_notional_quote,
        'forward_rate': trade.forward_rate,
        'maturity_date': trade.maturity_date if hasattr(trade, 'maturity_date') else None
    })

print("="*80)
print(f"HEDGE EXECUTION NOTIONALS (Tenor: {tenor})")
print("="*80)

print(f"\n{'Pair':<10} {'Direction':<10} {'Notional (Base)':>18} {'Notional (Quote)':>20} "
      f"{'Forward Rate':>14} {'Maturity'}")
print("-"*100)

total_notional_base = 0

for hedge_data in hedge_positions_manual:
    total_notional_base += hedge_data['trade_notional_base']
    
    # Extract currency from currency_pair
    pair = hedge_data['currency_pair']
    base_ccy = pair[:3]
    quote_ccy = pair[3:6]
    currency = quote_ccy if base_ccy == hedge_manager.home_currency else base_ccy
    
    maturity_str = hedge_data['maturity_date'].strftime('%Y-%m-%d') if hedge_data['maturity_date'] else 'N/A'
    
    print(f"{hedge_data['currency_pair']:<10} {hedge_data['direction']:<10} "
          f"${hedge_data['trade_notional_base']:>17,.0f} {hedge_data['trade_notional_quote']:>19,.0f} {currency} "
          f"{hedge_data['forward_rate']:>14.4f} {maturity_str}")

print("-"*100)
print(f"{'TOTAL':<10} {'':10} ${total_notional_base:>17,.0f}")

print("\n" + "="*80)
print("TRADE DIRECTION INTERPRETATION:")
print("  EURUSD/GBPUSD: SELL = Sell foreign currency (hedge depreciation risk)")
print("  USDJPY: BUY = Buy USD/Sell JPY (equivalent to hedging JPY exposure)")
print("="*80)

HEDGE EXECUTION NOTIONALS (Tenor: 3M)

Pair       Direction     Notional (Base)     Notional (Quote)   Forward Rate Maturity
----------------------------------------------------------------------------------------------------
EURUSD     SELL       $          678,577             702,598 EUR         1.0396 N/A
GBPUSD     SELL       $          264,500             331,048 GBP         1.2507 N/A
USDJPY     BUY        $                0                   0 JPY       155.5506 N/A
----------------------------------------------------------------------------------------------------
TOTAL                 $          943,077

TRADE DIRECTION INTERPRETATION:
  EURUSD/GBPUSD: SELL = Sell foreign currency (hedge depreciation risk)
  USDJPY: BUY = Buy USD/Sell JPY (equivalent to hedging JPY exposure)


## 9. Comprehensive Hedge Summary

Generate a single summary table combining all analysis.

In [11]:
# Generate comprehensive summary manually (generate_hedge_summary has a bug with list return type)
# Build summary from existing data

# Get the latest total NAV from nav_history
if portfolio_from_nav.nav_history is not None and 'Total' in portfolio_from_nav.nav_history.columns:
    latest_total_nav = portfolio_from_nav.nav_history['Total'].iloc[-1]
else:
    latest_total_nav = portfolio_from_nav.initial_nav

summary_data = []

for trade in trades:
    currency = trade.currency
    carry_obj = carry_results.get(currency)
    
    if carry_obj:
        carry_interpretation = 'BENEFIT' if carry_obj.is_benefit else 'COST'
    else:
        carry_interpretation = 'N/A'
    
    summary_data.append({
        'Currency': currency,
        'Pair': trade.currency_pair,
        'Exposure_NAV': trade.exposure_pf_home,
        'Weight_%': (trade.exposure_pf_home / latest_total_nav) * 100,
        'Spot': trade.spot_rate,
        f'Forward_{tenor}': trade.forward_rate,
        'Premium_bps': trade.forward_points,
        'Trade_Direction': trade.trade_direction,
        'Carry_bps': trade.carry_bps_ann if trade.carry_bps_ann is not None else 0,
        'Carry': carry_interpretation,
        'Recommended_Ratio_%': trade.recommended_ratio * 100,
        'Rationale': trade.rationale
    })

summary_df = pd.DataFrame(summary_data)

print("="*100)
print(f"COMPREHENSIVE HEDGE SUMMARY (Tenor: {tenor})")
print("="*100)
print()
print(summary_df.to_string(index=False))
print()
print("="*100)

COMPREHENSIVE HEDGE SUMMARY (Tenor: 3M)

Currency   Pair  Exposure_NAV  Weight_%     Spot  Forward_3M  Premium_bps Trade_Direction  Carry_bps   Carry  Recommended_Ratio_%           Rationale
     EUR EURUSD  1003711.6477   34.1033   1.0354      1.0396      42.4000            SELL   164.2514 BENEFIT              70.0000 Constant ratio: 70%
     GBP GBPUSD   472926.3240   16.0687   1.2516      1.2507      -8.5900            SELL   -27.5283    COST              70.0000 Constant ratio: 70%
     JPY USDJPY        0.0000    0.0000 157.2000    155.5506    -164.9400             BUY   420.8477 BENEFIT              50.0000 Constant ratio: 50%



## 10. Strategy-Based Hedging (NEW in v2.5.0)

Demonstrate the new Strategy Pattern implementation with different hedging strategies.

In [12]:
# Approach 1: Constant Ratio Strategy
print("="*80)
print("STRATEGY 1: CONSTANT RATIO")
print("="*80)

constant_strategy = ConstantRatioStrategy(
    ratios={'EUR': 0.95, 'GBP': 0.60},
    default_ratio=0.50,
    default_tenor='3M'
)

hedge_manager.set_strategy(constant_strategy)
constant_trades = hedge_manager.generate_trades()  # Uses strategy default tenor

print(f"Strategy: {constant_strategy.name()}")
print(f"\n{'Currency':<10} {'Ratio':>12} {'Rationale'}")
print("-"*50)
for trade in constant_trades:
    print(f"{trade.currency:<10} {trade.recommended_ratio*100:>11.1f}% {trade.rationale}")

print("\n" + "="*80)
print("STRATEGY 2: POLICY DRIVEN")
print("="*80)

# Approach 2: Policy Driven Strategy (replaces old HedgingPolicy)
policy_strategy = PolicyDrivenStrategy(
    policies={'EUR': 0.90, 'GBP': 0.70},
    default_ratio=0.50,
    default_tenor='3M'
)

hedge_manager.set_strategy(policy_strategy)
policy_trades = hedge_manager.generate_trades()

print(f"Strategy: {policy_strategy.name()}")
print(f"\n{'Currency':<10} {'Ratio':>12} {'Rationale'}")
print("-"*50)
for trade in policy_trades:
    print(f"{trade.currency:<10} {trade.recommended_ratio*100:>11.1f}% {trade.rationale}")

print("\n" + "="*80)
print("STRATEGY 3: CARRY COST WITH THRESHOLDS")
print("="*80)

# Approach 3: Carry Cost with custom thresholds
carry_strategy_custom = CarryCostStrategy(
    cost_thresholds={
        'EURUSD': (-150, 120),  # (min_bps, max_bps)
        'GBPUSD': (-100, 100)
    },
    default_tenor='3M'
)

hedge_manager.set_strategy(carry_strategy_custom)
carry_custom_trades = hedge_manager.generate_trades()

print(f"Strategy: {carry_strategy_custom.name()}")
print(f"Thresholds: EURUSD=(-150, 120), GBPUSD=(-100, 100)")
print(f"\n{'Currency':<10} {'Carry':>12} {'Ratio':>12} {'Rationale'}")
print("-"*80)
for trade in carry_custom_trades:
    carry_str = f"{trade.carry_bps_ann:+.1f} bps" if trade.carry_bps_ann is not None else "N/A"
    print(f"{trade.currency:<10} {carry_str:>12} {trade.recommended_ratio*100:>11.1f}% {trade.rationale}")

STRATEGY 1: CONSTANT RATIO
Strategy: Constant Ratio Strategy

Currency          Ratio Rationale
--------------------------------------------------
EUR               95.0% Constant ratio: 95%
GBP               60.0% Constant ratio: 60%
JPY               50.0% Constant ratio: 50%

STRATEGY 2: POLICY DRIVEN
Strategy: Policy Driven Strategy

Currency          Ratio Rationale
--------------------------------------------------
EUR               90.0% Policy-driven: 90%
GBP               70.0% Policy-driven: 70%
JPY               50.0% Default policy: 50% (no specific policy for JPY)

STRATEGY 3: CARRY COST WITH THRESHOLDS
Strategy: Carry Cost Based Strategy
Thresholds: EURUSD=(-150, 120), GBPUSD=(-100, 100)

Currency          Carry        Ratio Rationale
--------------------------------------------------------------------------------
EUR          +164.3 bps       100.0% Strong positive carry (>100 bps) (increased: benefit exceeds 120 bps threshold)
GBP           -27.5 bps        30.0% Small 

In [13]:
# Reset to default carry-cost strategy for remaining analysis
hedge_manager.set_strategy(CarryCostStrategy(default_tenor='3M'))
carry_trades = hedge_manager.generate_trades()

print("\n" + "="*80)
print("STRATEGY COMPARISON")
print("="*80)

# Create a mapping of currency to trades for easy lookup
constant_trades_map = {t.currency: t for t in constant_trades}
carry_trades_map = {t.currency: t for t in carry_trades}
policy_trades_map = {t.currency: t for t in policy_trades}
custom_trades_map = {t.currency: t for t in carry_custom_trades}

print(f"\n{'Currency':<10} {'Carry-Cost':>12} {'Constant':>12} {'Policy':>12} {'Custom Carry':>15}")
print("-"*80)

# Get all currencies from one of the trade sets
for trade in carry_trades:
    currency = trade.currency
    carry_ratio = carry_trades_map[currency].recommended_ratio * 100
    constant_ratio = constant_trades_map[currency].recommended_ratio * 100
    policy_ratio = policy_trades_map[currency].recommended_ratio * 100
    custom_ratio = custom_trades_map[currency].recommended_ratio * 100
    
    print(f"{currency:<10} {carry_ratio:>11.1f}% {constant_ratio:>11.1f}% "
          f"{policy_ratio:>11.1f}% {custom_ratio:>14.1f}%")


STRATEGY COMPARISON

Currency     Carry-Cost     Constant       Policy    Custom Carry
--------------------------------------------------------------------------------
EUR               90.0%        95.0%        90.0%          100.0%
GBP               30.0%        60.0%        70.0%           30.0%
JPY               90.0%        50.0%        50.0%           90.0%


## 11. Multi-Tenor Analysis

Compare hedging costs across different tenors.

In [14]:
# Analyze carry across multiple tenors
tenors = ['1M', '3M', '6M', '12M']
tenor_analysis = {}

for tenor_loop in tenors:
    tenor_analysis[tenor_loop] = {}
    for currency in exposures.keys():
        currency_pair = exposures[currency]['currency_pair']
        if currency_pair in forward_curves:
            try:
                # NEW API: Must pass foreign_currency parameter
                carry = hedge_manager.analyze_carry(
                    currency_pair=currency_pair,
                    foreign_currency=currency,
                    tenor=tenor_loop
                )
                tenor_analysis[tenor_loop][currency] = carry.carry_bps_ann
            except:
                tenor_analysis[tenor_loop][currency] = np.nan

# Create DataFrame
tenor_df = pd.DataFrame(tenor_analysis)

print("="*80)
print("MULTI-TENOR CARRY ANALYSIS")
print("="*80)
print()
print(tenor_df.to_string())
print()
print("Note: Values in annualized basis points (bps)")
print("      Positive = benefit, Negative = cost")
print("="*80)

MULTI-TENOR CARRY ANALYSIS

          1M       3M       6M      12M
EUR 149.5863 164.2514 186.4293 206.0073
GBP -32.3706 -27.5283 -27.4161 -24.5845
JPY 446.4981 420.8477 408.7917 390.0000

Note: Values in annualized basis points (bps)
      Positive = benefit, Negative = cost


## 12. Summary & Key Takeaways

In [15]:
print("="*80)
print("HEDGING WORKFLOW SUMMARY")
print("="*80)

print("\n1. PORTFOLIO SETUP")
print(f"   • Home Currency: {portfolio_from_nav.home_currency}")
print(f"   • Initial NAV: ${portfolio_from_nav.initial_nav:,.0f}")
print(f"   • Final NAV: ${nav_history_df['Total'].iloc[-1]:,.0f}")
print(f"   • Return: {portfolio_from_nav.total_return()*100:.2f}%")

print("\n2. FOREIGN CURRENCY EXPOSURES")
for currency in exposures.keys():
    exp_data = exposures[currency]
    print(f"   • {currency}: ${exp_data['nav']:,.0f} ({exp_data['weight']*100:.1f}%)")

print("\n3. FORWARD CARRY ANALYSIS (3M)")
for currency, carry in carry_results.items():
    interp = 'BENEFIT' if carry.is_benefit else 'COST'
    print(f"   • {carry.currency_pair} ({carry.trade_direction}): {carry.carry_bps_ann:+.1f} bps ({interp})")

print("\n4. HEDGE RECOMMENDATIONS (Strategy-Based)")
print(f"   Strategy Used: {hedge_manager.strategy.name()}")
for trade in trades:
    print(f"   • {trade.currency}: {trade.recommended_ratio*100:.0f}% hedge ({trade.rationale})")

print("\n5. HEDGE NOTIONALS FOR EXECUTION")
total_hedged = 0
for hedge_data in hedge_positions_manual:
    total_hedged += hedge_data['trade_notional_base']
    print(f"   • {hedge_data['currency_pair']}: {hedge_data['direction']} ${hedge_data['trade_notional_base']:,.0f}")
print(f"   • TOTAL NOTIONAL: ${total_hedged:,.0f}")

print("\n" + "="*80)
print("KEY TAKEAWAYS - v2.5.0 TO v2.7.0")
print("="*80)
print("""
✓ Strategy Pattern Implementation (v2.5.0):
  • HedgeManager now uses pluggable strategies for hedge ratio decisions
  • Four built-in strategies: CarryCostStrategy, ConstantRatioStrategy, 
    PolicyDrivenStrategy, QuantileBasedStrategy (NEW v2.7.0)
  • Runtime strategy switching via set_strategy() method
  • Easy to create custom strategies by extending HedgingStrategy ABC

✓ Enhanced Strategy Output (v2.7.0):
  • StrategyRecommendation dataclass with confidence, ratio_range, adjustment_factors
  • Strategies can now return rich decision context beyond just ratios
  • strategy_metadata flattened into trade_history columns for analysis
  • Access to full forward curve history via HedgingContext.forward_curves

✓ QuantileBasedStrategy (v2.7.0 Example):
  • Uses historical carry distributions for dynamic ratio decisions
  • Demonstrates context.carry_quantiles property (p10, p25, p50, p75, p90)
  • Shows how to create data-driven strategies with confidence levels
  • Formula: ratio = base_ratio + sensitivity * (percentile - 0.5)

✓ Clean API Design:
  • Removed boolean flags (cost_based, use_policies) for cleaner interface
  • Strategy parameter in constructor (defaults to CarryCostStrategy)
  • All hedge ratio logic encapsulated in strategy objects
  • Separation of concerns: HedgeManager orchestrates, strategies decide

✓ Backward Compatibility:
  • Default behavior unchanged (CarryCostStrategy auto-initialized)
  • analyze_carry() requires foreign_currency parameter for clarity
  • Carry logic unchanged: positive = benefit, negative = cost
  • Existing strategies work with both dict and StrategyRecommendation output

✓ Extensibility:
  • Add new strategies without modifying HedgeManager
  • Test strategies independently with mock data
  • Future-ready for optimization-based and ML-driven strategies

REFACTORED LOGIC:
• 4-step carry calculation: Asset/Liability → Convention → Trade Direction → Carry
• Trade direction explicitly determined based on exposure type
• Carry properly accounts for whether you're buying or selling the forward
• carry_bps_ann attribute name clarifies values are annualized

MIGRATION FROM OLD API:
# OLD (v2.4.x and earlier):
hm = HedgeManager(portfolio, curves)
recs = hm.recommend_hedge_ratios(tenor='3M', cost_based=True, use_policies=False)

# NEW (v2.5.0+):
strategy = CarryCostStrategy()
hm = HedgeManager(portfolio, curves, strategy=strategy)
trades = hm.generate_trades(tenor='3M')  # Use generate_trades() instead

# ADVANCED (v2.7.0+):
strategy = QuantileBasedStrategy(base_ratio=0.5, sensitivity=0.4)
hm = HedgeManager(portfolio, curves, strategy=strategy)
trades = hm.generate_trades()  # Returns enhanced output with confidence, metadata

NEXT STEPS:
• Implement rolling hedge strategy with automatic rebalancing
• Add P&L attribution analysis (hedged vs unhedged returns)
• Build optimization-based strategies (minimize cost, maximize return, etc.)
• Add transaction cost modeling (bid-ask spreads, commissions)
• Simulate hedge effectiveness over historical periods
• Create ML-based strategies using historical carry patterns
""")
print("="*80)

HEDGING WORKFLOW SUMMARY

1. PORTFOLIO SETUP
   • Home Currency: USD
   • Initial NAV: $994,163
   • Final NAV: $2,943,147
   • Return: 196.04%

2. FOREIGN CURRENCY EXPOSURES
   • EUR: $1,003,712 (34.1%)
   • GBP: $472,926 (16.1%)
   • JPY: $0 (0.0%)

3. FORWARD CARRY ANALYSIS (3M)
   • EURUSD (SELL): +164.3 bps (BENEFIT)
   • GBPUSD (SELL): -27.5 bps (COST)
   • USDJPY (BUY): +420.8 bps (BENEFIT)

4. HEDGE RECOMMENDATIONS (Strategy-Based)
   Strategy Used: Carry Cost Based Strategy
   • EUR: 70% hedge (Constant ratio: 70%)
   • GBP: 70% hedge (Constant ratio: 70%)
   • JPY: 50% hedge (Constant ratio: 50%)

5. HEDGE NOTIONALS FOR EXECUTION
   • EURUSD: SELL $678,577
   • GBPUSD: SELL $264,500
   • USDJPY: BUY $0
   • TOTAL NOTIONAL: $943,077

KEY TAKEAWAYS - v2.5.0 TO v2.7.0

✓ Strategy Pattern Implementation (v2.5.0):
  • HedgeManager now uses pluggable strategies for hedge ratio decisions
  • Four built-in strategies: CarryCostStrategy, ConstantRatioStrategy, 
    PolicyDrivenStrateg

In [1]:
%%capture
# Convert notebook to HTML with Plotly support
# Note: Run all cells first to ensure plots are in outputs
!jupyter nbconvert  \
"03_hedge_manager_demo.ipynb" \
--to html \
--template lab \
--output "03_hedge_manager_demo" \
--output-dir output \
--no-prompt